In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift')
print(sys.path)

In [ ]:
!pip install xplique timm opencv-python

In [ ]:
import shutil
shutil.copytree(
    '/content/drive/MyDrive/data/imageNet/imagenet_images',
    '/content/imagenet_images'
)

In [ ]:
from datasets import load_dataset
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from concept_helpers.DeepView_Craft import CraftTorchDV as Craft
from concept_helpers.DeepView_Craft import CraftTorchSupervised as CraftS
from concept_helpers.combined_crafts import CombinedCrafts

import urllib.request
import glob
import torch
import torch.nn as nn
from torchvision import transforms
import timm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from scipy.sparse.linalg import eigs
from sklearn.ensemble import  RandomForestClassifier

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import NMF
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

import random

from xplique.concepts.craft import BaseCraft, DisplayImportancesOrder, Factorization, Sensitivity
from sklearn.decomposition import non_negative_factorization
from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer

import os

device = 'cuda'

# loading any timm model
model = timm.create_model('nf_resnet50.ra2_in1k', pretrained=True)
model = model.to(device)

# processing
config = resolve_data_config({}, model=model)
transform = create_transform(**config)
to_pil = transforms.ToPILImage()

# cut the model in twop arts (as explained in the paper)
# first part is g(.) our 'input_to_latent' model, second part is h(.) our 'latent_to_logit' model
g = nn.Sequential(*(list(model.children())[:4]))  # input to penultimate layer
h = nn.Sequential(*(list(model.children())[4:]))  # penultimate layer to logits


with urllib.request.urlopen('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as f:
        imagenet_class_names = np.array(f.read().decode('utf-8').split('\n'))

def gen_images(filelist,folder_names,folder_name2class_id):
        for f in filelist:
            folder_name = f.split('/')[-2]
            if folder_name in folder_names:
                class_id = folder_name2class_id[folder_name]
                im = Image.open(f)
                if len(im.getbands()) == 3:
                    yield np.array(im.resize((224, 224))), class_id

# idd_folder = 'path/to/subset/of/imagenet'
idd_folder = '/content/imagenet_images'


# idd_folder_names = os.listdir('path/to/subset/of/imagenet')
idd_folder_names = os.listdir(idd_folder)
idd_class_names = idd_folder_names

idd_class_ids = [np.where(imagenet_class_names == class_name)[0][0] for class_name in idd_class_names]
folder_name2class_id = dict(zip(idd_folder_names, idd_class_ids))
filelist = glob.glob(f'{idd_folder}/*/*.jpg')


images, labels = zip(*gen_images(filelist,idd_folder_names,folder_name2class_id))
images, labels = np.array(images), np.array(labels)
preprocessed_images = torch.stack([transform(to_pil(img)) for img in images], 0)
print(preprocessed_images.shape)

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

def cosine_hungarian_loss(V1, V2):

    assert V1.shape == V2.shape
    assert len(V1.shape) == 2

    cost_matrix = 1 - cosine_similarity(V1, V2)

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    loss = cost_matrix[row_ind, col_ind].sum()

    loss = loss / len(V1)

    return float(loss)

def stability(v_drift_list):
  scores = []
  n = len(v_drift_list)
  for i in range(n):
    for j in range(i+1,n):
      scores.append(cosine_hungarian_loss(v_drift_list[i],v_drift_list[j]))

  return np.mean(scores) , np.std(scores)



def stability_across(result_dict, value_range):
    scores = []
    for val_a, val_b in combinations(value_range, 2):
        for va in result_dict[val_a]['v_drift_list']:
            for vb in result_dict[val_b]['v_drift_list']:
                scores.append(cosine_hungarian_loss(va, vb))
    return np.mean(scores), np.std(scores)

In [ ]:
# varying num of concepts
import gc

concept_range = [2,10,20,30, 40, 50]
runs = 25

full_size = 256
patch_size = 100


result_dict_concepts = {n_concepts: {
    "label_maps": [],
    "drift_ratios": [],
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for n_concepts in concept_range}



for n_concepts in concept_range:
    for run in range(runs):
        sample_ids = np.random.choice(len(preprocessed_images),500, False)

        sample_images = preprocessed_images[sample_ids]

        keys = idd_class_ids

        random.shuffle(keys)

        initial_labels = [0, 1, 2]
        random.shuffle(initial_labels)

        label_map = {keys[i]: initial_labels[i] for i in range(3)}

        for i in range(3, len(keys)):
            label_map[keys[i]] = random.randint(0, 2)

        result_dict_concepts[n_concepts]["label_maps"].append(label_map)

        labels_mapped = np.array([label_map[class_id] for class_id in labels])

        drift_labels = labels_mapped[sample_ids]

        label_2_idx = np.where(drift_labels == 2)[0]
        y_mixed = drift_labels.copy()
        y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

        sample_labels = y_mixed

        result_dict_concepts[n_concepts]["drift_ratios"].append({"BD": len(np.where(drift_labels == 0)[0]),
                                                        "AD": len(np.where(drift_labels == 1)[0]),
                                                        "Both": len(np.where(drift_labels == 2)[0])})

        h_craftdv = CraftS(input_to_latent_model=g,
                          latent_to_logit_model=h,
                          number_of_concepts=5,
                          inputs=sample_images,
                          labels=sample_labels,
                          batch_size=64,
                          patch_size=full_size,
                          device=device)

        patches, patch_act, train_labels = h_craftdv._extract_patches(sample_images, sample_labels )

        bd_indices = np.where(sample_labels != 1)[0]
        ad_indices = np.where(sample_labels != 0)[0]

        bd_fit = Craft(input_to_latent_model=g,
                      latent_to_logit_model=h,
                      number_of_concepts=n_concepts,
                      patch_size=patch_size,
                      batch_size=64,
                      device=device)
        print("Fitting Unsupervised Craft....")
        bd_crops, bd_crops_u, bd_w = bd_fit.fit(sample_images[bd_indices])

        ad_fit = Craft(input_to_latent_model=g,
                      latent_to_logit_model=h,
                      number_of_concepts=n_concepts,
                      patch_size=patch_size,
                      batch_size=64,
                      device=device)
        print("Fitting Unsupervised Craft....")
        ad_crops, ad_crops_u, ad_w = ad_fit.fit(sample_images[ad_indices])

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_concepts[n_concepts]["v_drift_list"].append(drift_basis.copy())

        drift_craft = CombinedCrafts(input_to_latent_model=g,
                                    latent_to_logit_model=h,
                                    number_of_concepts=len(drift_basis),
                                    inputs=sample_images,
                                    labels=sample_labels,
                                    basis = drift_basis,
                                    batch_size=64,
                                    patch_size=patch_size,
                                    device=device)
        print("Fitting Craft....")
        drift_craft.transform_all()

        X_clean = patch_act
        y_clean = train_labels

        localizer_model = Localizer()

        X_train_clean, X_test_clean, y_train, y_test = \
            train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

        print('Fitting Random Forest classifier...')
        localizer_model.fit(X_train_clean, y_train);
        print('Fitting complete.')

        localizer_bin_preds = localizer_model.l_predict(X_test_clean)

        result_dict_concepts[n_concepts]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

        image_drift_imp_l = [
            estimate_importance_helper_l(
                drift_craft, localizer_model, drift_basis,
                image, class_of_interest=localizer_bin_preds[i])
            for i, image in enumerate(X_test_clean)
        ]

        localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
        image_drift_imp_l_train = [
            estimate_importance_helper_l(
                drift_craft, localizer_model, drift_basis,
                image, class_of_interest=localizer_bin_train_preds[i])
            for i, image in enumerate(X_train_clean)
        ]
        concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

        result_dict_concepts[n_concepts]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
        result_dict_concepts[n_concepts]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
        result_dict_concepts[n_concepts]['recon_single'].append(
            accuracy_score(localizer_model.l_predict(recon_single), y_test))

        result_dict_concepts[n_concepts]['recon_single_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2*n_concepts)
        result_dict_concepts[n_concepts]['recon_all'].append(
            accuracy_score(localizer_model.l_predict(recon_all), y_test))

        result_dict_concepts[n_concepts]['recon_all_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_all), localizer_bin_preds))

        del h_craftdv, bd_fit, ad_fit, drift_craft
        gc.collect()
        print(f"concepts = {n_concepts} , run {run+1}")


In [ ]:
for n in concept_range:
    h_mean    = np.mean(result_dict_concepts[n]['drift_localizer'])
    h_std     = np.std(result_dict_concepts[n]['drift_localizer'])
    ht_mean   = np.mean(result_dict_concepts[n]['one_local_l'])
    ht_std    = np.std(result_dict_concepts[n]['one_local_l'])
    ht_lp_mean   = np.mean(result_dict_concepts[n]['one_local_l_lp'])
    ht_lp_std    = np.std(result_dict_concepts[n]['one_local_l_lp'])
    rs_mean   = np.mean(result_dict_concepts[n]['recon_single'])
    rs_std    = np.std(result_dict_concepts[n]['recon_single'])
    rs_lp_mean   = np.mean(result_dict_concepts[n]['recon_single_lp'])
    rs_lp_std    = np.std(result_dict_concepts[n]['recon_single_lp'])
    ra_mean   = np.mean(result_dict_concepts[n]['recon_all'])
    ra_std    = np.std(result_dict_concepts[n]['recon_all'])
    ra_lp_mean   = np.mean(result_dict_concepts[n]['recon_all_lp'])
    ra_lp_std    = np.std(result_dict_concepts[n]['recon_all_lp'])

    loss_mean, loss_std = stability(result_dict_concepts[n]['v_drift_list'])

    print(f"concepts: {n} "
          f"h = {h_mean:.3f}   {h_std:.3f}  "
          f"ht = {ht_mean:.3f}   {ht_std:.3f}  "
          f"ht_lp = {ht_lp_mean:.3f}   {ht_lp_std:.3f}  "
          f"rs = {rs_mean:.3f}   {rs_std:.3f}  "
          f"rs_lp = {rs_lp_mean:.3f}   {rs_lp_std:.3f}  "
          f"ra = {ra_mean:.3f}   {ra_std:.3f}  "
          f"ra_lp = {ra_lp_mean:.3f}   {ra_lp_std:.3f}  "
          f"stability = {loss_mean:.4f}   {loss_std:.4f}  ")

In [ ]:
n_concepts = 10
runs = 25

full_size = 256
patch_size = 100

data_points_range = [150,300,450,500,650,900]


result_dict_datapoints = {data_points: {
    "label_maps": [],
    "drift_ratios": [],
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for data_points in data_points_range}


for data_points in data_points_range:
    successful_runs = 0
    while successful_runs < runs:
        try:
            sample_ids = np.random.choice(len(preprocessed_images),data_points, False)

            sample_images = preprocessed_images[sample_ids]

            keys = idd_class_ids

            random.shuffle(keys)

            initial_labels = [0, 1, 2]
            random.shuffle(initial_labels)

            label_map = {keys[i]: initial_labels[i] for i in range(3)}

            for i in range(3, len(keys)):
                label_map[keys[i]] = random.randint(0, 2)

            labels_mapped = np.array([label_map[class_id] for class_id in labels])

            drift_labels = labels_mapped[sample_ids]

            label_2_idx = np.where(drift_labels == 2)[0]
            y_mixed = drift_labels.copy()
            y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

            sample_labels = y_mixed

            h_craftdv = CraftS(input_to_latent_model=g,
                              latent_to_logit_model=h,
                              number_of_concepts=5,
                              inputs=sample_images,
                              labels=sample_labels,
                              batch_size=64,
                              patch_size=full_size,
                              device=device)

            patches, patch_act, train_labels = h_craftdv._extract_patches(sample_images, sample_labels )

            bd_indices = np.where(sample_labels != 1)[0]
            ad_indices = np.where(sample_labels != 0)[0]

            bd_fit = Craft(input_to_latent_model=g,
                          latent_to_logit_model=h,
                          number_of_concepts=n_concepts,
                          patch_size=patch_size,
                          batch_size=64,
                          device=device)
            print("Fitting Unsupervised Craft....")
            bd_crops, bd_crops_u, bd_w = bd_fit.fit(sample_images[bd_indices])

            ad_fit = Craft(input_to_latent_model=g,
                          latent_to_logit_model=h,
                          number_of_concepts=n_concepts,
                          patch_size=patch_size,
                          batch_size=64,
                          device=device)
            print("Fitting Unsupervised Craft....")
            ad_crops, ad_crops_u, ad_w = ad_fit.fit(sample_images[ad_indices])

            drift_basis = np.vstack([bd_w, ad_w])

            drift_craft = CombinedCrafts(input_to_latent_model=g,
                                        latent_to_logit_model=h,
                                        number_of_concepts=len(drift_basis),
                                        inputs=sample_images,
                                        labels=sample_labels,
                                        basis = drift_basis,
                                        batch_size=64,
                                        patch_size=patch_size,
                                        device=device)
            print("Fitting Craft....")
            drift_craft.transform_all()

            X_clean = patch_act
            y_clean = train_labels

            localizer_model = Localizer()

            X_train_clean, X_test_clean, y_train, y_test = \
                train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

            print('Fitting Random Forest classifier...')
            localizer_model.fit(X_train_clean, y_train);
            print('Fitting complete.')

            localizer_bin_preds = localizer_model.l_predict(X_test_clean)

            drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

            image_drift_imp_l = [
                estimate_importance_helper_l(
                    drift_craft, localizer_model, drift_basis,
                    image, class_of_interest=localizer_bin_preds[i])
                for i, image in enumerate(X_test_clean)
            ]

            localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
            image_drift_imp_l_train = [
                estimate_importance_helper_l(
                    drift_craft, localizer_model, drift_basis,
                    image, class_of_interest=localizer_bin_train_preds[i])
                for i, image in enumerate(X_train_clean)
            ]
            concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

            result_dict_datapoints[data_points]["label_maps"].append(label_map)
            result_dict_datapoints[data_points]["drift_ratios"].append({"BD": len(np.where(drift_labels == 0)[0]),
                                                            "AD": len(np.where(drift_labels == 1)[0]),
                                                            "Both": len(np.where(drift_labels == 2)[0])})
            result_dict_datapoints[data_points]["v_drift_list"].append(drift_basis.copy())
            result_dict_datapoints[data_points]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

            result_dict_datapoints[data_points]['one_local_l'].append(
                local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
            result_dict_datapoints[data_points]['one_local_l_lp'].append(
                local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))

            recon_single = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
            result_dict_datapoints[data_points]['recon_single'].append(
                accuracy_score(localizer_model.l_predict(recon_single), y_test))
            result_dict_datapoints[data_points]['recon_single_lp'].append(
                accuracy_score(localizer_model.l_predict(recon_single), localizer_bin_preds))

            recon_all = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2*n_concepts)
            result_dict_datapoints[data_points]['recon_all'].append(
                accuracy_score(localizer_model.l_predict(recon_all), y_test))
            result_dict_datapoints[data_points]['recon_all_lp'].append(
                accuracy_score(localizer_model.l_predict(recon_all), localizer_bin_preds))

            successful_runs += 1
            print(f"data points = {data_points} , run {successful_runs}")

        except ValueError as e:
            print(f"Run übersprungen (data_points={data_points}): {e}")


In [ ]:
for n in data_points_range:
    h_mean    = np.mean(result_dict_datapoints[n]['drift_localizer'])
    h_std     = np.std(result_dict_datapoints[n]['drift_localizer'])
    ht_mean   = np.mean(result_dict_datapoints[n]['one_local_l'])
    ht_std    = np.std(result_dict_datapoints[n]['one_local_l'])
    ht_lp_mean   = np.mean(result_dict_datapoints[n]['one_local_l_lp'])
    ht_lp_std    = np.std(result_dict_datapoints[n]['one_local_l_lp'])
    rs_mean   = np.mean(result_dict_datapoints[n]['recon_single'])
    rs_std    = np.std(result_dict_datapoints[n]['recon_single'])
    rs_lp_mean   = np.mean(result_dict_datapoints[n]['recon_single_lp'])
    rs_lp_std    = np.std(result_dict_datapoints[n]['recon_single_lp'])
    ra_mean   = np.mean(result_dict_datapoints[n]['recon_all'])
    ra_std    = np.std(result_dict_datapoints[n]['recon_all'])
    ra_lp_mean   = np.mean(result_dict_datapoints[n]['recon_all_lp'])
    ra_lp_std    = np.std(result_dict_datapoints[n]['recon_all_lp'])

    loss_mean, loss_std = stability(result_dict_datapoints[n]['v_drift_list'])

    print(f"data points: {n} "
          f"h = {h_mean:.3f}   {h_std:.3f}  "
          f"ht = {ht_mean:.3f}   {ht_std:.3f}  "
          f"ht_lp = {ht_lp_mean:.3f}   {ht_lp_std:.3f}  "
          f"rs = {rs_mean:.3f}   {rs_std:.3f}  "
          f"rs_lp = {rs_lp_mean:.3f}   {rs_lp_std:.3f}  "
          f"ra = {ra_mean:.3f}   {ra_std:.3f}  "
          f"ra_lp = {ra_lp_mean:.3f}   {ra_lp_std:.3f}  "
          f"stability = {loss_mean:.4f}   {loss_std:.4f}  ")

In [ ]:
loss_mean, loss_std = stability_across(result_dict_datapoints, data_points_range)
print(f"Cross-data-points stability: {loss_mean:.4f}   {loss_std:.4f}")

In [ ]:


runs = 20
n_concepts = 10

patch_size_range = [64,100,128]

full_size = 256

result_dict_patchsize = {patch_size_n: {
    "label_maps": [],
    "drift_ratios": [],
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for patch_size_n in patch_size_range}

for patch_size_n in patch_size_range:

    patch_size = patch_size_n
    for run in range(runs):
        sample_ids = np.random.choice(len(preprocessed_images),500, False)

        sample_images = preprocessed_images[sample_ids]

        keys = idd_class_ids

        random.shuffle(keys)

        initial_labels = [0, 1, 2]
        random.shuffle(initial_labels)

        label_map = {keys[i]: initial_labels[i] for i in range(3)}

        for i in range(3, len(keys)):
            label_map[keys[i]] = random.randint(0, 2)

        result_dict_patchsize[patch_size_n]["label_maps"].append(label_map)

        labels_mapped = np.array([label_map[class_id] for class_id in labels])

        drift_labels = labels_mapped[sample_ids]

        label_2_idx = np.where(drift_labels == 2)[0]
        y_mixed = drift_labels.copy()
        y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

        sample_labels = y_mixed

        result_dict_patchsize[patch_size_n]["drift_ratios"].append({"BD": len(np.where(drift_labels == 0)[0]),
                                                        "AD": len(np.where(drift_labels == 1)[0]),
                                                        "Both": len(np.where(drift_labels == 2)[0])})

        h_craftdv = CraftS(input_to_latent_model=g,
                          latent_to_logit_model=h,
                          number_of_concepts=5,
                          inputs=sample_images,
                          labels=sample_labels,
                          batch_size=64,
                          patch_size=full_size,
                          device=device)

        patches, patch_act, train_labels = h_craftdv._extract_patches(sample_images, sample_labels )

        bd_indices = np.where(sample_labels != 1)[0]
        ad_indices = np.where(sample_labels != 0)[0]

        bd_fit = Craft(input_to_latent_model=g,
                      latent_to_logit_model=h,
                      number_of_concepts=n_concepts,
                      patch_size=patch_size_n,
                      batch_size=64,
                      device=device)
        print("Fitting Unsupervised Craft....")
        bd_crops, bd_crops_u, bd_w = bd_fit.fit(sample_images[bd_indices])

        ad_fit = Craft(input_to_latent_model=g,
                      latent_to_logit_model=h,
                      number_of_concepts=n_concepts,
                      patch_size=patch_size_n,
                      batch_size=64,
                      device=device)
        print("Fitting Unsupervised Craft....")
        ad_crops, ad_crops_u, ad_w = ad_fit.fit(sample_images[ad_indices])

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_patchsize[patch_size_n]["v_drift_list"].append(drift_basis.copy())

        drift_craft = CombinedCrafts(input_to_latent_model=g,
                                    latent_to_logit_model=h,
                                    number_of_concepts=len(drift_basis),
                                    inputs=sample_images,
                                    labels=sample_labels,
                                    basis = drift_basis,
                                    batch_size=64,
                                    patch_size=patch_size_n,
                                    device=device)
        print("Fitting Craft....")
        drift_craft.transform_all()

        X_clean = patch_act
        y_clean = train_labels

        localizer_model = Localizer()

        X_train_clean, X_test_clean, y_train, y_test = \
            train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

        print('Fitting Random Forest classifier...')
        localizer_model.fit(X_train_clean, y_train);
        print('Fitting complete.')

        localizer_bin_preds = localizer_model.l_predict(X_test_clean)

        result_dict_patchsize[patch_size_n]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

        image_drift_imp_l = [
            estimate_importance_helper_l(
                drift_craft, localizer_model, drift_basis,
                image, class_of_interest=localizer_bin_preds[i])
            for i, image in enumerate(X_test_clean)
        ]

        localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
        image_drift_imp_l_train = [
            estimate_importance_helper_l(
                drift_craft, localizer_model, drift_basis,
                image, class_of_interest=localizer_bin_train_preds[i])
            for i, image in enumerate(X_train_clean)
        ]
        concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

        result_dict_patchsize[patch_size_n]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
        result_dict_patchsize[patch_size_n]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
        result_dict_patchsize[patch_size_n]['recon_single'].append(
            accuracy_score(localizer_model.l_predict(recon_single), y_test))

        result_dict_patchsize[patch_size_n]['recon_single_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2*n_concepts)
        result_dict_patchsize[patch_size_n]['recon_all'].append(
            accuracy_score(localizer_model.l_predict(recon_all), y_test))

        result_dict_patchsize[patch_size_n]['recon_all_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_all), localizer_bin_preds))

        print(f"patch size = {patch_size_n} , run {run+1}")

In [ ]:
for n in patch_size_range:
    h_mean    = np.mean(result_dict_patchsize[n]['drift_localizer'])
    h_std     = np.std(result_dict_patchsize[n]['drift_localizer'])
    ht_mean   = np.mean(result_dict_patchsize[n]['one_local_l'])
    ht_std    = np.std(result_dict_patchsize[n]['one_local_l'])
    ht_lp_mean   = np.mean(result_dict_patchsize[n]['one_local_l_lp'])
    ht_lp_std    = np.std(result_dict_patchsize[n]['one_local_l_lp'])
    rs_mean   = np.mean(result_dict_patchsize[n]['recon_single'])
    rs_std    = np.std(result_dict_patchsize[n]['recon_single'])
    rs_lp_mean   = np.mean(result_dict_patchsize[n]['recon_single_lp'])
    rs_lp_std    = np.std(result_dict_patchsize[n]['recon_single_lp'])
    ra_mean   = np.mean(result_dict_patchsize[n]['recon_all'])
    ra_std    = np.std(result_dict_patchsize[n]['recon_all'])
    ra_lp_mean   = np.mean(result_dict_patchsize[n]['recon_all_lp'])
    ra_lp_std    = np.std(result_dict_patchsize[n]['recon_all_lp'])

    loss_mean, loss_std = stability(result_dict_patchsize[n]['v_drift_list'])

    print(f"patch_size: {n} "
          f"h = {h_mean:.3f}   {h_std:.3f}  "
          f"ht = {ht_mean:.3f}   {ht_std:.3f}  "
          f"ht_lp = {ht_lp_mean:.3f}   {ht_lp_std:.3f}  "
          f"rs = {rs_mean:.3f}   {rs_std:.3f}  "
          f"rs_lp = {rs_lp_mean:.3f}   {rs_lp_std:.3f}  "
          f"ra = {ra_mean:.3f}   {ra_std:.3f}  "
          f"ra_lp = {ra_lp_mean:.3f}   {ra_lp_std:.3f}  "
          f"stability = {loss_mean:.4f}   {loss_std:.4f}  ")

In [ ]:
loss_mean, loss_std = stability_across(result_dict_patchsize, patch_size_range)
print(f"Cross-patch-size stability: {loss_mean:.4f}   {loss_std:.4f}")

In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_datapoints_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['num data points', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for data_points in data_points_range:
        for method_name in method_names:
            row = [data_points, method_name] + result_dict_datapoints[data_points][method_name]
            writer.writerow(row)

In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_patchsize_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['patch size', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for patch_size_n in patch_size_range:
        for method_name in method_names:
            row = [patch_size_n, method_name] + result_dict_patchsize[patch_size_n][method_name]
            writer.writerow(row)

In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_n_concepts_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['n_concepts', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for n_concepts in concept_range:
        for method_name in method_names:
            row = [n_concepts, method_name] + result_dict_concepts[n_concepts][method_name]
            writer.writerow(row)

In [ ]:
import csv

with open('/content/drive/MyDrive/results/stability_n_concepts_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['n_concepts', 'stability_mean', 'stability_std'])
    for n_concepts in concept_range:
        loss_mean, loss_std = stability(result_dict_concepts[n_concepts]['v_drift_list'])
        writer.writerow([n_concepts, loss_mean, loss_std])

In [ ]:
import csv

with open('/content/drive/MyDrive/results/stability_datapoints_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['num data points', 'stability_mean', 'stability_std'])
    for data_points in data_points_range:
        loss_mean, loss_std = stability(result_dict_datapoints[data_points]['v_drift_list'])
        writer.writerow([data_points, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_datapoints, data_points_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])

In [ ]:
import csv

with open('/content/drive/MyDrive/results/stability_patchsizes_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['patch_size', 'stability_mean', 'stability_std'])
    for patch_size_n in patch_size_range:
        loss_mean, loss_std = stability(result_dict_patchsize[patch_size_n]['v_drift_list'])
        writer.writerow([patch_size_n, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_patchsize, patch_size_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])

In [ ]:
import csv

with open('/content/drive/MyDrive/results/results_datapoints_D2.csv') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows_650_900 = [row for row in reader if row[0] in ('650', '900')]

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_datapoints_D2_combined.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)

    for data_points in [150, 300, 450, 500]:
        for method_name in method_names:
            row = [data_points, method_name] + result_dict_datapoints[data_points][method_name]
            writer.writerow(row)

    for row in rows_650_900:
        writer.writerow(row)

In [ ]:
import numpy as np
for n in [150, 300, 450, 500]:
    loss_mean, loss_std = stability(result_dict_datapoints[n]['v_drift_list'])
    print(f"data points: {n}  stability = {loss_mean:.4f}   {loss_std:.4f}")

In [ ]:
import csv

with open('/content/drive/MyDrive/results/stability_datapoints_D2.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['num data points', 'stability_mean', 'stability_std'])
    for n in [150, 300, 450, 500]:
        loss_mean, loss_std = stability(result_dict_datapoints[n]['v_drift_list'])
        writer.writerow([n, loss_mean, loss_std])